# Phase 4.5: Model Evaluation

# Imports

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)

sys.path.insert(0, os.path.join('..', 'backend'))
from app.ml.feature_builder import MODEL_FEATURES, build_features_from_dataframe

# Paths

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'processed', 'cleaned_shot_logs.csv')
MODEL_PATH = os.path.join('..', 'backend', 'app', 'ml', 'models', 'shot_make_xgb_baseline.ubj')

print(DATA_PATH)
print(MODEL_PATH)

# Load Test Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

X = build_features_from_dataframe(df)
y = df['shot_made'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Total rows: {len(df):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Test make rate: {y_test.mean():.3f}')

X_test.head()

# Load Trained XGBoost Model

In [ ]:
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

print('Model loaded')

# Predict On Test Dataset

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

eval_df = df.loc[X_test.index].copy()
eval_df['actual_make'] = y_test.values
eval_df['predicted_make'] = y_pred
eval_df['predicted_p_make'] = y_prob

eval_df[[
    'shot_zone',
    'pressure_level',
    'shot_distance',
    'defender_distance',
    'actual_make',
    'predicted_make',
    'predicted_p_make',
]].head()

# Classification Report

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

print('\nClassification Report')
print(classification_report(y_test, y_pred, target_names=['Miss', 'Make'], zero_division=0))

# Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Miss', 'Make'],
    cmap='Blues',
    values_format='d',
)

plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print(f'True Misses:  {tn:,}')
print(f'False Makes:  {fp:,}')
print(f'False Misses: {fn:,}')
print(f'True Makes:   {tp:,}')

# ROC-AUC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'XGBoost AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

# Shot Zone Comparison

In [ ]:
def safe_auc(group):
    if group['actual_make'].nunique() < 2:
        return np.nan
    return roc_auc_score(group['actual_make'], group['predicted_p_make'])


def group_summary(data, group_col, order):
    rows = []

    for value in order:
        group = data[data[group_col] == value]

        rows.append({
            group_col: value,
            'shots': len(group),
            'actual_make_rate': group['actual_make'].mean(),
            'avg_predicted_p_make': group['predicted_p_make'].mean(),
            'predicted_make_rate': group['predicted_make'].mean(),
            'accuracy': accuracy_score(group['actual_make'], group['predicted_make']),
            'roc_auc': safe_auc(group),
        })

    return pd.DataFrame(rows).set_index(group_col).round(4)

In [ ]:
zone_order = ['Paint', 'Mid-Range', 'Three Point']
zone_summary = group_summary(eval_df, 'shot_zone', zone_order)

display(zone_summary)

zone_summary[['actual_make_rate', 'avg_predicted_p_make']].plot(kind='bar', figsize=(7, 4))
plt.title('Actual vs Predicted Make Rate by Shot Zone')
plt.xlabel('Shot Zone')
plt.ylabel('Rate')
plt.xticks(rotation=0)
plt.ylim(0, 0.75)
plt.tight_layout()
plt.show()

In [ ]:
zone_box_data = [
    eval_df.loc[eval_df['shot_zone'] == zone, 'predicted_p_make']
    for zone in zone_order
]

plt.figure(figsize=(8, 4))
plt.boxplot(zone_box_data, tick_labels=zone_order, patch_artist=True)
plt.title('Predicted P(make) by Shot Zone')
plt.xlabel('Shot Zone')
plt.ylabel('Predicted P(make)')
plt.tight_layout()
plt.show()

# Pressure Level Comparison

In [ ]:
pressure_order = ['Very Tight', 'Tight', 'Open', 'Very Open']
pressure_summary = group_summary(eval_df, 'pressure_level', pressure_order)

display(pressure_summary)

pressure_summary[['actual_make_rate', 'avg_predicted_p_make']].plot(kind='bar', figsize=(8, 4))
plt.title('Actual vs Predicted Make Rate by Pressure Level')
plt.xlabel('Pressure Level')
plt.ylabel('Rate')
plt.xticks(rotation=0)
plt.ylim(0, 0.75)
plt.tight_layout()
plt.show()

In [ ]:
pressure_box_data = [
    eval_df.loc[eval_df['pressure_level'] == pressure, 'predicted_p_make']
    for pressure in pressure_order
]

plt.figure(figsize=(8, 4))
plt.boxplot(pressure_box_data, tick_labels=pressure_order, patch_artist=True)
plt.title('Predicted P(make) by Pressure Level')
plt.xlabel('Pressure Level')
plt.ylabel('Predicted P(make)')
plt.tight_layout()
plt.show()

# What The Model Does Well

- Learns shot probability from real shot data.
- Gives a continuous `P(make)` value.
- Uses shot distance, defender distance, shot value, shot zone, and pressure level.
- Predicts different behavior for Paint, Mid-Range, and Three Point shots.

# What The Model Struggles With

- It does not know player skill.
- It does not know exact court coordinates.
- It does not know game context.
- `shot_angle` is currently not useful because missing values become `0`.

# Is It Good Enough For Phase 4 Prototype

Yes. The model is good enough for the Phase 4 prototype because ROC-AUC is above random guessing and the zone behavior makes basketball sense.

# Better Than Phase 3 Rule-Based Model

- Phase 3 used fixed manual rules.
- Phase 4 learns from real data.
- Phase 4 gives measurable performance metrics.
- Phase 4 can learn feature interactions that rules may miss.